Ali Issabayev is responsible for this code

### Pixel Segmentation (Base Model)

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/home/default/Desktop/project")
DATASET_DIR = PROJECT_ROOT / "Dataset 4cats" / "dataset_yolo_4cats_seg"
DATA_YAML = DATASET_DIR / "taco.yaml"
DATA_YAML_OVERSAMPLED = DATASET_DIR / "taco_oversampled.yaml"

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR, "->", DATASET_DIR.exists())
print("Data YAML:", DATA_YAML, "->", DATA_YAML.exists())
print("Oversampled YAML:", DATA_YAML_OVERSAMPLED, "->", DATA_YAML_OVERSAMPLED.exists())

Project root: /home/default/Desktop/project
Dataset dir: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg -> True
Data YAML: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/taco.yaml -> True
Oversampled YAML: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/taco_oversampled.yaml -> True


In [2]:
# Check basic structure
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    path = DATASET_DIR / sub
    print(f"{sub:20} -> exists: {path.exists()}, num files: {len(list(path.glob('*')))}")

images/train         -> exists: True, num files: 1275
images/val           -> exists: True, num files: 225
labels/train         -> exists: True, num files: 1275
labels/val           -> exists: True, num files: 225


In [3]:
from itertools import islice

label_files = sorted((DATASET_DIR / "labels/train").glob("*.txt"))
print("Number of train label files:", len(label_files))

# Show a few sample lines
for lf in islice(label_files, 3):
    print("\nFile:", lf.name)
    with lf.open("r") as f:
        for i, line in zip(range(3), f):
            parts = line.strip().split()
            print(" line:", line.strip())
            print("  -> tokens:", len(parts))
            if len(parts) > 5:
                print("  -> looks like SEGMENTATION (class + polygon coords)")
            else:
                print("  -> looks like BBOX only (not segmentation)")

Number of train label files: 1275

File: 0_000006.txt
 line: 1 0.364997 0.604197 0.369551 0.586140 0.368900 0.573450 0.357189 0.550024 0.350033 0.531479 0.337671 0.509029 0.336370 0.490483 0.340273 0.470473 0.344177 0.461201 0.338321 0.437286 0.341574 0.420693 0.348731 0.400683 0.360442 0.375305 0.375407 0.354807 0.387118 0.330893 0.387768 0.285505 0.382563 0.168863 0.378009 0.160078 0.370202 0.149341 0.370852 0.134700 0.374756 0.109322 0.364346 0.100049 0.366949 0.082967 0.376057 0.075159 0.395576 0.066374 0.422251 0.061981 0.447625 0.061981 0.472349 0.062958 0.493819 0.068814 0.510085 0.074671 0.515290 0.086384 0.512687 0.094192 0.508783 0.102001 0.515290 0.116154 0.521796 0.132260 0.521796 0.143485 0.514639 0.155686 0.513338 0.175695 0.516591 0.192777 0.527001 0.258175 0.532856 0.297218 0.547170 0.329429 0.573845 0.355295 0.595966 0.381162 0.603774 0.391410 0.610280 0.407028 0.611581 0.417765 0.610930 0.426061 0.610280 0.431430 0.614183 0.439727 0.618738 0.447535 0.621991 0.459736 0

In [4]:
import yaml

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Train path:", data_cfg.get("train"))
print("Val path:", data_cfg.get("val"))
print("Classes (names):")
for k, v in data_cfg.get("names", {}).items():
    print(f"  {k}: {v}")

Train path: images/train
Val path: images/val
Classes (names):
  0: plastic
  1: glass
  2: paper
  3: unsorted


In [ ]:
# Possible model size: "yolov8n-seg.pt", "yolov8s-seg.pt", "yolov8m-seg.pt"...

from ultralytics import YOLO
MODEL_NAME = "yolov8s-seg.pt" # for speed, the smallest model first

model = YOLO(MODEL_NAME)
print(model)

YOLO(
  (model): SegmentationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_runnin

In [ ]:
# Use the original data (with 4 supercats)
use_oversampled = False
data_yaml_path = DATA_YAML_OVERSAMPLED if use_oversampled else DATA_YAML
print("Using data yaml:", data_yaml_path)

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=8,
    project="taco_seg_runs",
    name="yolov8s_seg_4cats",
    pretrained=True,
    verbose=True
)

Using data yaml: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/taco.yaml
Ultralytics 8.3.233 🚀 Python-3.10.18 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/taco.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=Fals

In [8]:
# Use the best weights saved during training
best_weights = Path("taco_seg_runs") / "yolov8s_seg_4cats2" / "weights" / "best.pt"
print("Best weights path:", best_weights, "->", best_weights.exists())

model_val = YOLO(str(best_weights))

metrics = model_val.val(
    data=str(data_yaml_path),
    imgsz=640
)

print("Segmentation mAP50:", metrics.results_dict.get("metrics/seg/mAP50", "N/A"))
print("Segmentation mAP50-95:", metrics.results_dict.get("metrics/seg/mAP50-95", "N/A"))

Best weights path: taco_seg_runs/yolov8s_seg_4cats2/weights/best.pt -> True
Ultralytics 8.3.233 🚀 Python-3.10.18 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)


YOLOv8s-seg summary (fused): 85 layers, 11,781,148 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6439.2±1446.9 MB/s, size: 2222.7 KB)
val: Scanning /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/labels/val.cache... 225 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 225/225 367.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.2it/s 12.4s0.7s
                   all        225        727      0.418      0.305      0.268      0.197      0.464      0.278      0.262      0.169
               plastic        168        322      0.475      0.509      0.455      0.344      0.526      0.472      0.451        0.3
                 glass         15         27      0.408      0.185      0.145     0.0989      0.509      0.185      0.145     0.0567
                 paper         47         71      0.283      0.352   

In [ ]:
# Use oversampled data
use_oversampled = True
data_yaml_path = DATA_YAML_OVERSAMPLED if use_oversampled else DATA_YAML
print("Using data yaml:", data_yaml_path)

results = model.train(
    data=str(data_yaml_path),
    epochs=100,        
    imgsz=640,
    batch=8,           
    project="taco_seg_runs",
    name="yolov8s_seg_4cats",
    pretrained=True,   
    verbose=True
)

Using data yaml: /home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/taco_oversampled.yaml
Ultralytics 8.3.233 🚀 Python-3.10.18 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/default/Desktop/project/Dataset 4cats/dataset_yolo_4cats_seg/taco_oversampled.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosa

  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  8                  -1  1   1838080  ultralytics.nn.modules.block.C2f             [512, 512, 1, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  1    591360  ultralytics.nn.modules.block.C2f             [768,

KeyboardInterrupt: 